In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
data = pd.read_csv("all_participants_data.csv", index_col="Participant")
filteredData = data.drop(columns=['median_arousal', 'median_valence'])
targetData = data['median_arousal']

indexes = data.index

scaler = StandardScaler()
scaledArray = scaler.fit_transform(filteredData)
scaledData = pd.DataFrame(scaledArray, index=indexes)

print(scaledData)

                  0         1         2         3         4         5    \
Participant                                                               
64          -1.665550 -1.018178  0.162946  0.311937 -0.094151 -1.458478   
64          -1.547531 -0.988172  0.422245  0.623738 -0.008691 -1.384106   
64          -0.977216 -0.724075 -0.146381  0.229324  0.442454 -1.105813   
64          -0.218558  0.150993  0.458450  0.784200  1.168580 -0.211696   
64           0.075725  0.221303  0.027481  0.288399  0.747573 -0.061314   
...               ...       ...       ...       ...       ...       ...   
30           1.063190  1.484203  0.569686  0.643383  1.267968  1.515181   
30           0.939504  1.424867  1.099175  1.180893  1.795443  1.487396   
30           0.976282  1.368941  1.408992  1.329795  2.022155  1.450898   
30           1.191407  1.334019  1.951544  1.708168  2.398909  1.506098   
30           1.280602  1.261865  0.550702  0.383950  1.952202  1.441351   

                  6     

## Linear Regression Model

In [3]:
#Linear Regression Model

from sklearn.linear_model import LinearRegression

model1 = LinearRegression()


## Neural Network Regressor

In [4]:
#Neural Network Regressor

from sklearn.neural_network import MLPRegressor

model2 = MLPRegressor(
    hidden_layer_sizes=(16, 8),   # 2 hidden layers
    activation='relu',                 # activation function
    solver='adam',                     # optimizer
    max_iter=100,                      # max training iterations
    random_state=42
)

## LSTM Regressor

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,        # use the passed value
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, _ = self.lstm(x)
        last = lstm_out[:, -1, :]
        return self.fc(last)
    

model3 = LSTMRegressor(1, 64, 3, 1)


In [6]:
indexes = data.index
participants = set(indexes)

print(participants)

dataSubsets = [data.loc[participant] for participant in participants]


{64, 65, 34, 37, 39, 41, 42, 45, 46, 16, 19, 21, 23, 56, 25, 26, 28, 30}


In [7]:
from scipy.stats import pearsonr
import numpy as np

def CCcoefficient(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))

    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2)
    return ccc

# **Model Tests**

In [8]:
# testing model 1

participants = list(set(indexes))

results = []

X = scaledData
Y = targetData

for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = X.loc[trainIdx], Y.loc[trainIdx]
    Xtest, Ytest = X.loc[testIdx], Y.loc[testIdx]

    model1.fit(Xtrain, Ytrain)

    Ypred = model1.predict(Xtest)

    results.append((pearsonr(np.array(Ytest).ravel(), np.array(Ypred).ravel())[0], CCcoefficient(Ytest, Ypred)))

print(results)
    

[(0.6332975208189173, 0.6074461087302238), (0.5385252939786885, 0.4413296008001172), (0.518527752599073, 0.2882028919808958), (0.5760845582711356, 0.4636279720649508), (0.2205149958244541, 0.20557169595645844), (0.5285298604008858, 0.20150514554991356), (0.548388959002055, 0.4567637485256043), (0.6144319963628319, 0.5213030553604867), (0.612554826637379, 0.590159155339129), (0.5111725962872049, 0.477661705274307), (0.5921464571323531, 0.4743715068629638), (0.5541323373156137, 0.4070355722506708), (0.5787642890321133, 0.5136055998881693), (0.5378718489549857, 0.49495867267059496), (0.7216543012259565, 0.6468173475669509), (0.651045015224772, 0.49924966791243824), (0.5842276025218647, 0.5129930963118385), (0.6275345751271759, 0.36328982688879613)]


In [13]:
# testing model 2

participants = list(set(indexes))

results = []

X = scaledData
Y = targetData

for participant in participants:
    trainIdx = [p for p in participants if p != participant]
    testIdx = participant

    Xtrain, Ytrain = X.loc[trainIdx], Y.loc[trainIdx]
    Xtest, Ytest = X.loc[testIdx], Y.loc[testIdx]

    model2.fit(Xtrain, Ytrain)

    Ypred = model2.predict(Xtest)

    results.append((pearsonr(np.array(Ytest).ravel(), np.array(Ypred).ravel())[0], CCcoefficient(Ytest, Ypred)))

print(results)

[(0.552055177550851, 0.536219626507567), (0.5045294973659089, 0.49855581419922995), (0.4020260599792436, 0.27387449081661), (0.41475239173456874, 0.32501422520414414), (0.050995083994000864, 0.0441341197516764), (0.4606498751791355, 0.25753093565046303), (0.3748247757753346, 0.34982124935184605), (0.5150208685101803, 0.49307002050744947), (0.4582364460172545, 0.4540876995691389), (0.44154713785705646, 0.44129134119296337), (0.38114438140392176, 0.2905555735385297), (0.43202272634065597, 0.41122581714452666), (0.4719758587184867, 0.4434685134485786), (0.41962389783376264, 0.39070997262388096), (0.5754388372977192, 0.5577769839553876), (0.47235386943118834, 0.4135487314596742), (0.4634665474296205, 0.42174945056637325), (0.521005046624111, 0.34867958368746815)]


In [ ]:
# testing model 3

from torch.utils.data import TensorDataset, DataLoader

X = scaledData.to_numpy()
Y = targetData.to_numpy()

participants = np.array(list(set(indexes)))
n_features = scaledData.shape[1]           # number of columns in X

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(Y, dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = []
for participant in participants:
    test_mask = indexes == participant
    train_mask = ~test_mask

    Xtrain = X_tensor[train_mask].view(-1, 1, n_features).to(device)
    Ytrain = y_tensor[train_mask].view(-1, 1).to(device)
    Xtest  = X_tensor[test_mask].view(-1, 1, n_features).to(device)
    Ytest  = y_tensor[test_mask].view(-1, 1).to(device)

    model = LSTMRegressor(
        input_size=n_features,
        hidden_size=16,
        num_layers=2,
        output_size=1
    ).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    train_loader = DataLoader(
        TensorDataset(Xtrain, Ytrain),
        batch_size=64,      
        shuffle=True
    )

    for epoch in range(10):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            y_pred = model(xb)
            loss = criterion(y_pred, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        Ypred_test = model(Xtest).cpu().numpy().ravel()
        Ytrue      = Ytest.cpu().numpy().ravel()

    results.append((
        pearsonr(Ytrue, Ypred_test)[0],
        CCcoefficient(Ytrue, Ypred_test)
    ))

print(results)
        



[(0.46994638, 0.4197096572053832), (0.42531446, 0.35912005866652263), (-0.09416791, -0.025236394602113715), (0.17048517, 0.16365938826211246), (0.26142398, 0.14539775764983126), (0.30281612, 0.2849172265511025), (0.412476, 0.4074725050688116), (0.38351953, 0.3659009426370165), (0.21172373, 0.17897341385662738), (0.3114427, 0.296580377465173), (0.57448304, 0.5632107939009805), (0.23554726, 0.22180699561994532), (0.55102324, 0.5236342981796714), (0.3960172, 0.3804916089751886), (0.672876, 0.3803546901611395), (0.40177256, 0.29258329228286595), (0.42148668, 0.4131960650510137), (0.35721672, 0.2955567083581978)]
